# Авито: кандидатогенерация для поиска услуг (Recall@50)

**Задача:** по поисковому запросу отобрать из корпуса (189 212 объявлений) до 50 кандидатов так, чтобы
в них попали объявления, которые пользователь в итоге выбрал. Метрика — Recall@50.

**Итоговый подход — гибридный скоринг всего корпуса**, итоговый скор объявления для запроса:

$$
score = \underbrace{BM25_{title} + w_D BM25_{desc} + w_P BM25_{params}}_{\text{лексика}}
      + w_F\,BM25_{params}(\text{фильтры})
      + w_E \cos(e5_q, e5_d)
      + w_L \log P(\text{лок. объявления}\mid\text{лок. поиска})
      + w_M \log P(\text{подкатегория}\mid\text{запрос})
      + w_C\,\text{память}
      + w_{pop}\log(1+\text{отзывы})
$$

| компонента | откуда | зачем |
|---|---|---|
| BM25 по заголовку / описанию / параметрам | корпус | точные совпадения слов, стемминг Snowball |
| BM25 фильтров запроса по параметрам объявления | `search_infm_params_text` | мягкий учёт фильтров («Вид услуги», «Тип услуги»…) |
| плотный поиск e5 | `intfloat/multilingual-e5-small` | синонимы и перефразы, которых BM25 не видит |
| априор локации | train | 83% выборов — в той же локации; «зонтичные» локации (Москва+МО, вся Россия) раскладываются на города по статистике train |
| априор подкатегории | train, kNN по текстам запросов | запрос «автоподбор» почти всегда ведёт в одну-две подкатегории |
| память | train, kNN по текстам запросов | объявления корпуса, которые уже выбирали по похожим запросам |
| популярность | число отзывов | пользователи чаще выбирают объявления с отзывами |

Веса подобраны покоординатным перебором на локальной валидации (см. раздел 6).

**Использованные open-source инструменты:** pandas, numpy, scipy, scikit-learn (TF-IDF, CountVectorizer),
PyStemmer (стеммер Snowball для русского), sentence-transformers + модель `intfloat/multilingual-e5-small` (MIT),
PyTorch. Внешних API нет: модель скачивается с HuggingFace один раз и дальше работает локально.
При разработке использовался AI-ассистент (Claude) — для обсуждения идей, написания и ревью кода.

## 0. Окружение и настройки

In [1]:
# pip install pandas pyarrow numpy scipy scikit-learn PyStemmer torch sentence-transformers
import os, re, time
from functools import lru_cache

import numpy as np
import pandas as pd
import scipy.sparse as sp
import Stemmer
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize as l2norm
from sentence_transformers import SentenceTransformer

DATA_DIR = 'data'            # папка с train.parquet, benchmark_queries.parquet, benchmark_items.parquet
CACHE_DIR = 'cache'          # сюда кладутся эмбеддинги корпуса, чтобы не считать их повторно
os.makedirs(CACHE_DIR, exist_ok=True)

K = 50                       # размер выдачи
SEED = 42
BLOCK = 250                  # сколько запросов скорим за раз (плотная матрица 189k x BLOCK float32 ≈ 190 МБ)
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
E5_MODEL = 'intfloat/multilingual-e5-small'

# Ключ «запроса»: в бенчмарке запрос = текст + локация + фильтры + категория
QKEY = ['search_query', 'search_location_id', 'search_is_delivery_search',
        'search_infm_params_text', 'search_category']

# Веса компонент скора — подобраны на валидации (раздел 6)
WEIGHTS = {'D': 1.5, 'P': 0, 'F': 0.6, 'L': 5, 'M': 1, 'pop': 1, 'E': 60, 'C': 10}
print(DEVICE)

mps


## 1. Данные и что в них важно

Коротко о том, что нашлось при разведочном анализе (цифры пересчитываются ячейкой ниже):
* в train ~500k пар «запрос → выбранное объявление», но **лишь ~10% объявлений корпуса встречаются в train** — просто
  «запомнить» пары нельзя, нужен настоящий поиск по тексту;
* **все 2452 текста запросов бенчмарка уникальны, и только ~37% из них встречались в train**;
* **локация — сильнейший сигнал**: в 83% пар локация объявления совпадает с локацией поиска. Остальное — в основном
  «зонтичные» локации поиска (например, 107620 — судя по данным, Москва с областью; 621540 — вся Россия),
  в которых самих объявлений нет, а выбирают объявления из дочерних городов;
* у запроса обычно 1 релевантное объявление (в среднем ~1.03).

In [2]:
def load_data():
    tr = pd.read_parquet(f'{DATA_DIR}/train.parquet')
    bq = pd.read_parquet(f'{DATA_DIR}/benchmark_queries.parquet')
    it = pd.read_parquet(f'{DATA_DIR}/benchmark_items.parquet').reset_index(drop=True)
    for df in (tr, bq, it):                     # id — строки, ничего не приводим к числам
        for c in ('item_id', 'query_id'):
            if c in df: df[c] = df[c].astype(str)
    for c in ('search_query', 'search_infm_params_text'):
        tr[c] = tr[c].fillna(''); bq[c] = bq[c].fillna('')
    return tr, bq, it

train, bench_q, items = load_data()
print('train', train.shape, 'bench', bench_q.shape, 'items', items.shape)
print('доля объявлений корпуса, встречавшихся в train:', round(items.item_id.isin(set(train.item_id)).mean(), 3))
print('доля текстов бенчмарка, встречавшихся в train:', round(bench_q.search_query.isin(set(train.search_query)).mean(), 3))
print('train: локация поиска == локации объявления:', round((train.search_location_id == train.item_location_id).mean(), 3))
print('бенчмарк: локации поиска, в которых нет ни одного объявления корпуса:',
      round((~bench_q.search_location_id.isin(set(items.item_location_id))).mean(), 3))

train (497673, 19) bench (2452, 6) items (189212, 14)


доля объявлений корпуса, встречавшихся в train: 0.096


доля текстов бенчмарка, встречавшихся в train: 0.37
train: локация поиска == локации объявления: 0.831
бенчмарк: локации поиска, в которых нет ни одного объявления корпуса: 0.174


## 2. Нормализация текста и BM25

Тексты: нижний регистр, `ё→е`, токены из букв и цифр, стемминг Snowball (так «телевизоров» и «телевизор» совпадут).

BM25 считаю сам на разреженных матрицах: для каждого поля строится матрица весов «документ × термин», скор запроса —
произведение этой матрицы на бинарный вектор запроса. Поля (заголовок, описание, параметры) индексируются отдельно,
потому что вклад совпадения в заголовке и в длинном описании разный — веса полей подобраны на валидации.

In [3]:
_stemmer = Stemmer.Stemmer('russian')
_tok_re = re.compile(r'[a-zа-я0-9]+')

@lru_cache(maxsize=2_000_000)
def _stem(w):
    return _stemmer.stemWord(w)

def normalize(s):
    return (s or '').lower().replace('ё', 'е')

def tokenize(s):
    return [_stem(w) for w in _tok_re.findall(normalize(s))]


class FieldBM25:
    '''BM25 одного поля. vocabulary общий для всех полей, чтобы матрица запроса была одна.'''
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, docs, vocabulary):
        self.cv = CountVectorizer(analyzer=tokenize, vocabulary=vocabulary, dtype=np.float32)
        tf = self.cv.transform(docs).tocsr()
        n = tf.shape[0]
        dl = np.asarray(tf.sum(1)).ravel()                      # длины документов
        avgdl = max(dl.mean(), 1.0)
        df = np.bincount(tf.indices, minlength=tf.shape[1])     # документная частота термина
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))
        # классическая формула BM25: idf * tf*(k1+1) / (tf + k1*(1 - b + b*dl/avgdl))
        denom = tf.data + self.k1 * (1 - self.b + self.b * dl[rows] / avgdl)
        data = idf[tf.indices] * tf.data * (self.k1 + 1) / denom
        self.W = sp.csr_matrix((data.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape)
        return self

    def query_matrix(self, queries):
        q = self.cv.transform(queries)
        q.data[:] = 1.0                    # повтор слова в коротком запросе ничего не должен усиливать
        return q.tocsc().astype(np.float32)

    def scores(self, qmat):                # -> плотная матрица «документ × запрос»
        return (self.W @ qmat.T).toarray().astype(np.float32)

In [4]:
t0 = time.time()
fields = {
    'title':  items.item_title_raw.fillna(''),
    'params': items.item_infm_params_text.fillna(''),
    'desc':   items.item_description_raw.fillna(''),
}
vocab = CountVectorizer(analyzer=tokenize).fit(
    fields['title'] + ' ' + fields['params'] + ' ' + fields['desc']).vocabulary_
bm25 = {name: FieldBM25().fit(docs, vocab) for name, docs in fields.items()}
print('словарь', len(vocab), 'терминов, индексация заняла', round(time.time() - t0), 'с')

словарь 387819 терминов, индексация заняла 38 с


## 3. Плотный поиск: multilingual-e5-small

BM25 не знает, что «автоподбор» и «помощь в выборе автомобиля» — одно и то же. Для этого беру небольшую
мультиязычную модель эмбеддингов `intfloat/multilingual-e5-small` (MIT, 118M параметров, работает на CPU/MPS).
Документ = заголовок + первые 400 символов описания (дальше обычно контакты и «воду»), длина до 128 токенов.
По требованию модели документы кодируются с префиксом `passage: `, запросы — с `query: `.
Эмбеддинги корпуса кэшируются в `cache/emb_items.npy` (на M-процессоре ~4 минуты).

In [5]:
e5 = SentenceTransformer(E5_MODEL, device=DEVICE)
e5.max_seq_length = 128
emb_path = f'{CACHE_DIR}/emb_items.npy'
if os.path.exists(emb_path):
    item_emb = np.load(emb_path)
else:
    docs = ('passage: ' + items.item_title_raw.fillna('') + '. '
            + items.item_description_raw.fillna('').str.slice(0, 400)).tolist()
    order = np.argsort([len(d) for d in docs])          # сортировка по длине: меньше паддинга, быстрее
    E = e5.encode([docs[i] for i in order], batch_size=256, normalize_embeddings=True,
                  show_progress_bar=True, convert_to_numpy=True)
    item_emb = np.empty_like(E); item_emb[order] = E
    item_emb = item_emb.astype(np.float16)
    np.save(emb_path, item_emb)
item_emb_t = torch.tensor(item_emb.astype(np.float32))
print(item_emb.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/740 [00:00<?, ?it/s]

(189212, 384)


## 4. Сигналы из train: локация, подкатегория, память

Все три считаются функцией `fit_train_stats(df)` от переданного куска train. Это важно для честной валидации:
на валидации статистики строятся **без** валидационных запросов, на финальном прогоне — по всему train.

**Априор локации** $P(\text{loc объявления} \mid \text{loc поиска})$ — сглаженная частота из train; к локации
поиска добавляется псевдосчётчик `beta` (объявление из той же локации подходит, даже если по ней мало статистики),
остальным — маленький `eps`, чтобы очень релевантное по тексту объявление из другого города всё же могло попасть в топ.

**Априор подкатегории и память** опираются на поиск ближайших текстов запросов из train (TF-IDF по стеммам +
биграммам стемм + символьным 3–5-граммам, косинус). Для нового запроса берём 30 соседей с весами sim²:
* распределение подкатегорий `item_microcat_id`, которые выбирали по соседям → $\log P(\text{microcat}\mid q)$;
* объявления **из корпуса**, которые выбирали по соседям → признак «память».

Символьные n-граммы помогают с опечатками и формами слов («ремонт  пк» ↔ «ремонт пк», «скупка телевизоров» ↔ «скупка телевизора»).

In [6]:
item_pos = {x: i for i, x in enumerate(items.item_id)}
N_ITEMS = len(items)
pop_feat = np.log1p(items.item_rating_reviews_count.fillna(0).values).astype(np.float32)[:, None]


def bigram_analyzer(s):
    t = tokenize(s)
    return t + [a + '_' + b for a, b in zip(t, t[1:])]


class TrainStats:
    def __init__(self, df, beta=5.0, eps=1e-4, n_neighbors=30):
        self.beta, self.eps, self.k = beta, eps, n_neighbors
        # --- локации
        self.loc_cnt = df.groupby(['search_location_id', 'item_location_id']).size()
        self.loc_tot = df.groupby('search_location_id').size()
        all_locs = np.unique(np.r_[items.item_location_id, df.item_location_id, df.search_location_id])
        self.loc_code = {l: i for i, l in enumerate(all_locs)}
        self.item_loc_code = items.item_location_id.map(self.loc_code).values
        # --- тексты запросов train и их kNN-индекс
        self.texts = df.search_query.unique()
        tid = {t: i for i, t in enumerate(self.texts)}
        self.vec_w = TfidfVectorizer(analyzer=bigram_analyzer, sublinear_tf=True)
        self.vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True,
                                     preprocessor=normalize, min_df=2)
        self.X = l2norm(sp.hstack([self.vec_w.fit_transform(self.texts),
                                   self.vec_c.fit_transform(self.texts)]).tocsr())
        # текст -> распределение подкатегорий (строки нормированы)
        mcs = np.unique(np.r_[items.item_microcat_id, df.item_microcat_id])
        mci = {m: i for i, m in enumerate(mcs)}
        c = df.groupby('search_query').item_microcat_id.value_counts().rename('c').reset_index()
        TM = sp.csr_matrix((c.c.values.astype(np.float32), (c.search_query.map(tid), c.item_microcat_id.map(mci))),
                           shape=(len(self.texts), len(mcs)))
        self.TM = sp.csr_matrix(TM.multiply(1 / np.asarray(TM.sum(1))))
        self.item_mc = items.item_microcat_id.map(mci).values
        # текст -> выбранные объявления корпуса («память»)
        d = df[df.item_id.isin(item_pos)]
        self.TI = sp.csr_matrix((np.ones(len(d), np.float32), (d.search_query.map(tid), d.item_id.map(item_pos))),
                                shape=(len(self.texts), N_ITEMS))

    def loc_logprior(self, locs):
        '''-> N x nq: log P(локация объявления | локация поиска)'''
        out = np.empty((N_ITEMS, len(locs)), np.float32)
        for j, L in enumerate(locs):
            p = np.full(len(self.loc_code) + 1, self.eps, np.float32)
            t = self.loc_tot.get(L, 0)
            if t:
                c = self.loc_cnt.loc[L]
                p[[self.loc_code[x] for x in c.index]] += c.values / (t + self.beta)
            p[self.loc_code.get(L, -1)] += self.beta / (t + self.beta)   # -1 = «чужая» ячейка, если локации нет нигде
            out[:, j] = np.log(p)[self.item_loc_code]
        return out

    def neighbor_weights(self, qtexts):
        '''-> разреженная матрица nq x n_texts с весами sim^2 для k ближайших текстов train'''
        Z = l2norm(sp.hstack([self.vec_w.transform(qtexts), self.vec_c.transform(qtexts)]).tocsr())
        S = (Z @ self.X.T).toarray()
        top = np.argpartition(-S, self.k, axis=1)[:, :self.k]
        w = np.take_along_axis(S, top, 1) ** 2
        return sp.csr_matrix((w.ravel(), (np.repeat(np.arange(len(qtexts)), self.k), top.ravel())),
                             shape=(len(qtexts), len(self.texts)))

    def microcat_and_memory(self, qtexts, eps=1e-3):
        W = self.neighbor_weights(qtexts)
        P = (W @ self.TM).toarray() / (np.asarray(W.sum(1)) + 1e-9)            # nq x n_microcat
        mc = np.log(P + eps)[:, self.item_mc].T.astype(np.float32)             # N x nq
        mem = np.log1p((W @ self.TI).toarray() * 5).T.astype(np.float32)       # N x nq
        return mc, mem

## 5. Скоринг и выдача топ-50

Для блока запросов считаются все компоненты как плотные матрицы «объявление × запрос» и складываются с весами.
Скорится весь корпус — 189k объявлений это немного, и так не теряются кандидаты на отдельном этапе отсева.
Топ-50 берётся через `argpartition` и сортируется по скору (на метрику порядок не влияет, но так выдачу удобнее читать).

In [7]:
def score_block(q, stats, w=None):
    '''q — DataFrame с колонками search_*; возвращает матрицу скоров N x len(q) и словарь компонент.'''
    w = w or WEIGHTS
    texts = q.search_query.tolist()
    Q = bm25['title'].query_matrix(texts)
    Fq = bm25['params'].query_matrix(q.search_infm_params_text.tolist())
    qe = torch.tensor(e5.encode(['query: ' + t for t in texts], normalize_embeddings=True))
    mc, mem = stats.microcat_and_memory(texts)
    comp = {
        'T': bm25['title'].scores(Q), 'D': bm25['desc'].scores(Q), 'P': bm25['params'].scores(Q),
        'F': bm25['params'].scores(Fq), 'E': (item_emb_t @ qe.T).numpy(),
        'L': stats.loc_logprior(q.search_location_id.values), 'M': mc, 'C': mem,
    }
    S = comp['T'] + sum(w[k] * comp[k] for k in 'DPFELMC') + w['pop'] * pop_feat
    return S, comp


def retrieve(q, stats, w=None, k=K):
    '''-> dict query_id -> список из k item_id (по убыванию скора)'''
    ids = items.item_id.values
    out = {}
    for s in range(0, len(q), BLOCK):
        sub = q.iloc[s:s + BLOCK]
        S, _ = score_block(sub, stats, w)
        top = np.argpartition(-S, k, axis=0)[:k]
        for j, qid in enumerate(sub.query_id.values):
            t = top[:, j][np.argsort(-S[top[:, j], j])]
            out[qid] = list(ids[t])
    return out

## 6. Локальная валидация

Чтобы до отправки понимать качество, строю валидацию, которая **повторяет устройство бенчмарка**:
* беру группы train (ключ запроса как в бенчмарке), у которых **все** выбранные объявления лежат в корпусе бенчмарка —
  тогда Recall@50 считается по тому же корпусу из 189k объявлений, что и на платформе;
* по одной группе на текст запроса (в бенчмарке тексты уникальны), 3000 запросов;
* в бенчмарке только ~37% текстов встречались в train, поэтому для ~63% валидационных текстов **удаляю из train все
  строки с этим текстом** — такие запросы для модели «новые». Первая версия валидации этого не делала, в ней 82% текстов
  были «знакомыми», и она завышала качество;
* статистики (`TrainStats`) строятся только по оставшейся части train — утечки нет.

Веса `WEIGHTS` подобраны на этой валидации покоординатным подъёмом (по одному весу, по сетке, два прохода).
Ячейку с подбором оставил закомментированной — она идёт ~30 минут.

In [8]:
def recall_at_k(pred, rel, k=K):
    return float(np.mean([len(set(pred.get(q, [])[:k]) & r) / len(r) for q, r in rel.items()]))


def make_split(tr, n_val=3000, seen_share=0.37, seed=SEED):
    tr = tr.copy()
    tr['in_corpus'] = tr.item_id.isin(item_pos)
    g = tr.groupby(QKEY, sort=False).agg(n=('item_id', 'size'), nc=('in_corpus', 'sum')).reset_index()
    g = g[g.nc == g.n].sample(frac=1, random_state=seed).drop_duplicates('search_query')
    val = g.head(n_val).reset_index(drop=True)
    val['query_id'] = [f'val{i:013d}' for i in range(len(val))]
    rng = np.random.RandomState(seed)
    unseen = set(val.search_query[rng.rand(len(val)) > seen_share])
    rel = tr.merge(val[QKEY + ['query_id']], on=QKEY).groupby('query_id').item_id.apply(set).to_dict()
    fit = tr.merge(val[QKEY], on=QKEY, how='left', indicator=True)
    fit = fit[(fit._merge == 'left_only') & ~fit.search_query.isin(unseen)].drop(columns=['_merge', 'in_corpus'])
    return fit.reset_index(drop=True), val[['query_id'] + QKEY], rel


fit_df, val_q, val_rel = make_split(train)
val_stats = TrainStats(fit_df)
print('val запросов', len(val_q), '| доля текстов, виденных в train:',
      round(val_q.search_query.isin(set(fit_df.search_query)).mean(), 3))

val запросов 3000 | доля текстов, виденных в train: 0.29


In [9]:
t0 = time.time()
val_pred = retrieve(val_q, val_stats)
print('Recall@50 на валидации:', round(recall_at_k(val_pred, val_rel), 4), '| время', round(time.time() - t0), 'с')

Recall@50 на валидации: 0.9219 | время 22 с


### Вклад компонент (абляция)

Выключаю по одной компоненте (вес = 0) и смотрю, сколько теряет Recall@50.

In [10]:
abl = {'все компоненты': recall_at_k(val_pred, val_rel)}
names = {'L': 'локация', 'M': 'подкатегория', 'E': 'e5', 'D': 'BM25 описание', 'F': 'фильтры', 'pop': 'популярность', 'C': 'память'}
for k, name in names.items():
    abl[f'без: {name}'] = recall_at_k(retrieve(val_q, val_stats, {**WEIGHTS, k: 0}), val_rel)
abl['только BM25 заголовок+описание'] = recall_at_k(
    retrieve(val_q, val_stats, {k: (WEIGHTS['D'] if k == 'D' else 0) for k in WEIGHTS}), val_rel)
pd.Series(abl).round(4).to_frame('Recall@50')

,Recall@50
все компоненты,0.9219
без: локация,0.4577
без: подкатегория,0.9009
без: e5,0.9137
без: BM25 описание,0.8667
без: фильтры,0.9129
без: популярность,0.9164
без: память,0.9186
только BM25 заголовок+описание,0.3853


In [11]:
# Подбор весов (долго, ~30 мин). Результат уже записан в WEIGHTS.
# grid = dict(E=[0,10,20,40,80], C=[0,1,3,10], pop=[.5,1,2,3], M=[.5,1,2,4], L=[2,3,5], D=[.5,1,1.5], P=[0,.2,.5], F=[.1,.3,.6,1])
# w = dict(WEIGHTS)
# for _ in range(2):
#     for k, vals in grid.items():
#         res = {v: recall_at_k(retrieve(val_q, val_stats, {**w, k: v}), val_rel) for v in vals}
#         w[k] = max(res, key=res.get); print(k, res)

## 7. Анализ ошибок

In [12]:
v = val_q.copy()
v['recall'] = [len(set(val_pred[q]) & val_rel[q]) / len(val_rel[q]) for q in v.query_id]
v['text_seen'] = v.search_query.isin(set(fit_df.search_query))
v['loc_in_corpus'] = v.search_location_id.isin(set(items.item_location_id))
v['has_filter'] = v.search_infm_params_text.str.len() > 0
v['n_words'] = v.search_query.str.split().str.len().clip(upper=5)
for col in ['text_seen', 'loc_in_corpus', 'has_filter', 'n_words']:
    display(v.groupby(col).recall.agg(['mean', 'size']).round(3))

,mean,size
text_seen,,
False,0.918,2130
True,0.930,870


,mean,size
loc_in_corpus,,
False,0.843,507
True,0.938,2493


,mean,size
has_filter,,
False,0.919,1246
True,0.924,1754


,mean,size
n_words,,
1,0.916,274
2,0.927,1185
3,0.928,902
4,0.918,443
5,0.878,196


In [13]:
# Примеры промахов: запрос и что пользователь выбрал на самом деле
it_idx = items.set_index('item_id')
miss = v[v.recall == 0].sample(12, random_state=1)
rows = []
for _, r in miss.iterrows():
    tgt = next(iter(val_rel[r.query_id]))
    rows.append(dict(query=r.search_query, filters=r.search_infm_params_text[:40],
                     chosen_title=it_idx.at[tgt, 'item_title_raw'],
                     same_loc=it_idx.at[tgt, 'item_location_id'] == r.search_location_id,
                     our_top1=it_idx.at[val_pred[r.query_id][0], 'item_title_raw']))
pd.DataFrame(rows)

,query,filters,chosen_title,same_loc,our_top1
0,чистка колодца от частного лица,Кто оказывает услуги Частный исполнитель,Ремонт и чистка колодцев,False,Чистка и ремонт колодцев
1,снять база отдыха,,Гостевой дом на берегу Оби,False,База отдыха на Алтае с бассейном и анимацией
2,клоун на праздник,,Спектакль на ваше мероприятие,True,Аниматоры Клоуны на детский праздник
3,снять дом в селе нижне озерное,"Вид услуги Сад, благоустройство",Аренда бани с чаном,False,"Проведение семейного отдыха, праздников"
4,сулакский каньон домики,"Вид услуги Праздники, мероприятия",Домик для отдыха семейный по суточно,False,Сулакский каньон катер
5,домик в аренду,"Вид услуги Праздники, мероприятия",Сдаю в аренду территорию для вашего праздника,False,"Аренда беседки с бассейном, мангал, гостевой двор"
6,костоправ мануальщик,"Тип услуги СПА-услуги, массаж Вид услуги","Остеопат, врaч рeaбилитолог, ткм, кст",False,"Костоправ (мануалка, массаж)"
7,помощь в оформлении земли,,Помогу получить земельный участок у государства,False,Оформление детских пособий
8,аренда катка,,Аренда дорожного катка,False,Аренда катка
9,установка опор лэп,Тип услуги Электрика Участие в пилоте CP,"Электрик, Электромонтажные работы",False,"Установка столбов, опор лэп, подключение эл.сетям"


**Что видно по ошибкам** (подробнее — в README):
* **Локация — главный источник промахов.** ~60% промахов — объявления не из локации поиска. Хуже всего «зонтичные» локации
  (Москва+МО, вся Россия, СПб+ЛО), где самих объявлений нет: Recall там ~0.84 против ~0.94. Пример: «бурение скважины» →
  «Бурение скважин на воду» — текст совпадает идеально, но объявление из другого города проигрывает по априору.
  Что сделал: вместо жёсткого фильтра — мягкий априор $P(loc_{item}\mid loc_{search})$ по train с псевдосчётчиками;
  вес подобран на валидации (жёстче — теряем такие случаи, мягче — топ забивается объявлениями из других городов).
  Пробовал добавить географическую близость по координатам — прироста нет (+0.0003, в пределах шума), не включил.
* **Лексический разрыв** («аренда краскопульта» → «Прокат инструмента», «замена батарейки в ключе» → «Автоключи,
  программирование») — BM25 не видит связи. Частично закрывают e5 и априор подкатегории.
* **Длинные запросы (5+ слов)** — Recall ниже (~0.88): BM25 требует совпадения многих слов, часть из которых — топонимы
  («сантехник метро раменки»). Помогает e5 и то, что BM25 суммирует по словам, а не требует всех.
* **Шум в разметке** — часть «выборов» почти не связана с запросом («торт 3 шоколада» → «Торт Медовик на заказ»,
  «романтическое свидание» → «Семейный отдых с баней»). Такие случаи кандидатогенерация честно ловить не должна.


## 8. Финальный прогон на бенчмарке

Статистики строятся уже по **всему** train, веса — те же, что на валидации.

In [14]:
t0 = time.time()
full_stats = TrainStats(train)
bench_pred = retrieve(bench_q, full_stats)
answer = pd.DataFrame({'query_id': bench_q.query_id.values,
                       'answer': [' '.join(bench_pred[q]) for q in bench_q.query_id.values]})
answer.to_csv('answer.csv', index=False)
print('готово за', round(time.time() - t0), 'с')

готово за 19 с


In [15]:
# Проверки формата перед отправкой
chk = pd.read_csv('answer.csv', dtype=str)
all_items = set(items.item_id)
assert list(chk.columns) == ['query_id', 'answer']
assert len(chk) == len(bench_q) and set(chk.query_id) == set(bench_q.query_id) and chk.query_id.is_unique
assert chk.query_id.str.len().eq(16).all()
lists = chk.answer.str.split(' ')
assert lists.map(len).le(50).all() and lists.map(lambda l: len(l) == len(set(l))).all()
assert lists.map(lambda l: all(x in all_items and re.fullmatch(r'[0-9a-f]{16}', x) for x in l)).all()
print('формат ок; объявлений в строке:', lists.map(len).min(), '–', lists.map(len).max())
chk.head()

формат ок; объявлений в строке: 50 – 50


,query_id,answer
0,70DfDUpwjxB4lzFd,14d535285758af89 6fc4db453f873556 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,1c6d2079e07f4497 422d3ffdd5bbf626 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b d8fce513e4f000a7 168a9207e80b...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 67b2cd847669eb36 da18e5119a94...
4,YgHcM9MVbxKnxD1e,1a62634394544781 9c41fe789f98d702 615c73ea4c3b...
